In [ ]:
#Figure 3I-K & Figure 5D-F_Feeding-evoked (phasic) and tonic DA release

import os
import csv
import numpy as np
import pandas as pd
import tdt
import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"]     = "none"
plt.rcParams["font.family"]      = "sans-serif"
plt.rcParams["font.sans-serif"]  = ["Arial", "DejaVu Sans", "Malgun Gothic"]

# ──────────────────────────────────────────────────────────────────────────────
# 0) 사용자 설정
# ──────────────────────────────────────────────────────────────────────────────
save_dir        = "/path/to/your/TDT_data"

GRAB_CH         = "_465A"
ISO_CH          = "_405A"
DOWNSAMPLE      = 10
PELLET_CODE     = 2003     # pellet biting (t = 0)

# Phasic (z-score) 설정 (단위: 초, pellet biting = 0)
BASELINE_START  = -330     # z-score baseline
BASELINE_END    = -30
PRE_SEC         = 60       # mean trace 구간 -60 ~ +120 s
POST_SEC        = 120
PEAK_START      = 0        # peak 검출용 CSV 구간 0 ~ 30 s
PEAK_END        = 30

# Tonic 설정: pellet biting 기준 -300 ~ -120 s (3 min) 구간의 Mean465 / Mean405
TONIC_START     = -300
TONIC_END       = -120

# Mean trace Y축: 모든 그룹을 같은 범위로 고정해 저장
# (논문 그림에서는 이 SVG를 기준으로 그룹별 축 범위를 SVG 상에서 비율에 맞춰 조정)
TRACE_YLIM      = (-4.13, 12.43)

# ──────────────────────────────────────────────────────────────────────────────
# 1) Cohorts (Before / After: 같은 마우스끼리 같은 순서로 입력)
# ──────────────────────────────────────────────────────────────────────────────
ctrl_before_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]
ctrl_after_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]

stress_before_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]
stress_after_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]

gicno_before_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]
gicno_after_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]

vglut2_rs_before_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]
vglut2_rs_after_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]

vglut2_ctrl_before_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]
vglut2_ctrl_after_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]

gfp_rs_before_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]
gfp_rs_after_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]

gfp_ctrl_before_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]
gfp_ctrl_after_paths = [
    os.path.join(save_dir, "Mouse#", "Name of file"),
]

# (CSV 그룹명, 그래프 제목, 색, Before 경로, After 경로)
COHORTS = [
    ("Ctrl",        "Ctrl",            "forestgreen", ctrl_before_paths,        ctrl_after_paths),         # Fig 3
    ("Stress",      "Stress",          "royalblue",   stress_before_paths,      stress_after_paths),       # Fig 3
    ("GiCNO",       "Gi+CNO",          "goldenrod",   gicno_before_paths,       gicno_after_paths),        # Fig 3
    ("VGLUT2_RS",   "vGluT2 O/E + RS", "purple",      vglut2_rs_before_paths,   vglut2_rs_after_paths),    # Fig 5
    ("VGLUT2_Ctrl", "vGluT2 O/E Ctrl", "teal",        vglut2_ctrl_before_paths, vglut2_ctrl_after_paths),  # Fig 5
    ("GFP_RS",      "GFP + RS",        "crimson",     gfp_rs_before_paths,      gfp_rs_after_paths),       # Fig 5
    ("GFP_Ctrl",    "GFP Ctrl",        "gray",        gfp_ctrl_before_paths,    gfp_ctrl_after_paths),     # Fig 5
]

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2) 핵심 함수
# ──────────────────────────────────────────────────────────────────────────────
def block_mean(x, n):
    """n개 샘플씩 겹치지 않게 평균 (다운샘플링)"""
    m = len(x) // n * n
    return x[:m].reshape(-1, n).mean(axis=1)


def process_recording(path):
    """녹화 1개 처리.
    Phasic: 다운샘플 → 405 nm linear fit → ΔF/F → z-score (baseline -330~-30 s), -60~+120 s 구간 반환
    Tonic : raw 465/405 신호의 -300~-120 s 구간 평균 (Mean465 / Mean405)"""
    blk  = tdt.read_block(path)
    F465 = blk.streams[GRAB_CH].data.astype(float)
    F405 = blk.streams[ISO_CH].data.astype(float)
    fs   = blk.streams[GRAB_CH].fs

    note   = blk.epocs["Note"]
    onsets = note.onset[note.data == PELLET_CODE]
    if onsets.size == 0:
        raise KeyError(f"pellet code {PELLET_CODE} not found")
    pellet_t = float(onsets[0])

    # Phasic: ΔF/F → z-score
    F465_ds = block_mean(F465, DOWNSAMPLE)
    F405_ds = block_mean(F405, DOWNSAMPLE)
    time_ds = block_mean(np.arange(len(F465)) / fs, DOWNSAMPLE)

    a, b = np.polyfit(F405_ds, F465_ds, 1)
    fit  = a * F405_ds + b
    dFF  = (F465_ds - fit) / fit
    tr   = time_ds - pellet_t

    base = dFF[(tr >= BASELINE_START) & (tr <= BASELINE_END)]
    z    = (dFF - base.mean()) / base.std(ddof=1)

    win = (tr >= -PRE_SEC) & (tr <= POST_SEC)

    # Tonic: raw 신호 평균 비율
    i0 = max(int((pellet_t + TONIC_START) * fs), 0)
    i1 = min(int((pellet_t + TONIC_END) * fs), len(F465))
    m465, m405 = F465[i0:i1].mean(), F405[i0:i1].mean()

    return {"z": z[win], "t": tr[win], "Mean465": m465, "Mean405": m405}


def process_cohort(paths):
    """마우스별 z-score trace를 가장 짧은 길이에 맞춰 (n_mice, T) 행렬로 정리"""
    recs = []
    for p in paths:
        print(f"  {os.path.basename(os.path.dirname(p))} / {os.path.basename(p)}")
        recs.append(process_recording(p))
    L = min(len(r["z"]) for r in recs)
    return {
        "z":       np.vstack([r["z"][:L] for r in recs]),
        "t":       recs[0]["t"][:L],
        "mouse":   [os.path.basename(os.path.dirname(p)) for p in paths],
        "rec":     [os.path.basename(p) for p in paths],
        "Mean465": [r["Mean465"] for r in recs],
        "Mean405": [r["Mean405"] for r in recs],
    }

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3) 모든 코호트 처리 (녹화당 1회 읽기)
# ──────────────────────────────────────────────────────────────────────────────
data = {}
for key, _, _, before_paths, after_paths in COHORTS:
    print(f"=== {key} ===")
    data[(key, "before")] = process_cohort(before_paths)
    data[(key, "after")]  = process_cohort(after_paths)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 4) Figure 3I & 5D,E: Mean ± SEM trace (Before 검정, After 색)
# ──────────────────────────────────────────────────────────────────────────────
def mean_sem(z):
    """마우스 간 mean, SEM (SD는 ddof=0)"""
    n = np.sum(np.isfinite(z), axis=0)
    return np.nanmean(z, axis=0), np.nanstd(z, axis=0, ddof=0) / np.sqrt(np.maximum(n, 1))

for key, title, color, _, _ in COHORTS:
    zb, za = data[(key, "before")]["z"], data[(key, "after")]["z"]
    t = data[(key, "before")]["t"]
    L = min(zb.shape[1], za.shape[1], len(t))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for z, c, label in ((zb, "black", "Before"), (za, color, "After")):
        m, sem = mean_sem(z[:, :L])
        ax.plot(t[:L], m, color=c, linewidth=2, label=label)
        ax.fill_between(t[:L], m - sem, m + sem, color=c, alpha=0.2)
    ax.axvline(0, color="red", linestyle=":", label="pellet (0 s)")
    ax.set_xlim(-PRE_SEC, POST_SEC)
    ax.set_ylim(TRACE_YLIM)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Z-score")
    ax.set_title(f"{title} – Z-score (Mean ± SEM)")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False)
    plt.tight_layout()

    fname = title.replace(" ", "_").replace("+", "plus").replace("/", "_")
    out_svg = os.path.join(save_dir, f"Zscore_{fname}_MeanOnly.svg")
    plt.savefig(out_svg, dpi=300)
    plt.close(fig)
    print(f"[SAVED] {out_svg}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 5) Figure 3I,J & 5D-F (phasic): 0~30 s z-score CSV
#    → 마우스별 0~30 s 최대값(peak), 2nd/1st 비율(R_DA-phasic) 계산에 사용
# ──────────────────────────────────────────────────────────────────────────────
t_ref = data[("Ctrl", "before")]["t"]
idx   = np.flatnonzero((t_ref >= PEAK_START) & (t_ref <= PEAK_END))

phasic = {"Time_s": t_ref[idx]}
for key, *_ in COHORTS:
    for phase in ("before", "after"):
        d = data[(key, phase)]
        for mouse, z in zip(d["mouse"], d["z"][:, idx]):
            phasic[f"{mouse}_{phase}"] = z

out_phasic = os.path.join(save_dir, "zscore_0_30s_ALL_COHORTS.csv")
pd.DataFrame(phasic).to_csv(out_phasic, index=False)
print(f"CSV 저장: {out_phasic}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 6) Figure 3K & 5F (tonic): -300 ~ -120 s Mean465 / Mean405 CSV
#    → 2nd/1st 비율(R_DA-tonic) 계산에 사용
# ──────────────────────────────────────────────────────────────────────────────
out_tonic = os.path.join(save_dir, f"tonic_Mean465_Mean405_win_{TONIC_START}_{TONIC_END}.csv")
with open(out_tonic, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Group", "Phase", "MouseID", "Recording", "Mean465", "Mean405", "Mean465/Mean405"])
    for key, *_ in COHORTS:
        for phase in ("before", "after"):
            d = data[(key, phase)]
            for mouse, rec, m465, m405 in zip(d["mouse"], d["rec"], d["Mean465"], d["Mean405"]):
                writer.writerow([key, phase, mouse, rec, m465, m405, m465 / m405])
print(f"CSV 저장: {out_tonic}")